Entraîner un réseau de neurones convolutifs (CNN) pour cette tâche est disproportionné et sous-optimal. Le problème relève de la recherche d'instances par similarité locale (gérer des recadrages ou des translations) et non de la classification globale de catégories.

On privilégie les algorithmes de détection de points d'intérêt visuels (Local Feature Matching), comme SIFT (Scale-Invariant Feature Transform). Cette approche est mathématiquement conçue pour extraire des points invariants à l'échelle, à la rotation et aux changements d'illumination, ce qui permet de retrouver un fragment d'image (portrait) au sein d'une image plus large (paysage).

L'algorithme détecte des points singuliers et calcule un descripteur (un vecteur) pour chacun. La comparaison entre les images s'effectue en cherchant les descripteurs les plus proches. Pour filtrer les faux positifs, on applique systématiquement le test de ratio de Lowe.

Soit d1​ la distance spatiale entre un descripteur de l'image A et son voisin le plus proche dans l'image B, et d2​ la distance avec le second voisin le plus proche. Le point d'intérêt est considéré comme une correspondance valide si :
d2​d1​​<τ

où τ est un seuil de sélectivité (généralement 0.7). Si le total des correspondances valides entre deux images dépasse un certain seuil, elles sont identifiées comme une paire.

In [6]:
import cv2
import hashlib
from pathlib import Path

def calculer_empreinte_md5(chemin_fichier):
    # Calcul du hash MD5
    with open(chemin_fichier, "rb") as f:
        return hashlib.md5(f.read()).hexdigest()

def redimensionner_image(img, max_dim=800):
    # Redimensionnement proportionnel pour alléger SIFT
    h, w = img.shape
    if max(h, w) <= max_dim:
        return img
    facteur = max_dim / float(max(h, w))
    nouvelle_taille = (int(w * facteur), int(h * facteur))
    return cv2.resize(img, nouvelle_taille, interpolation=cv2.INTER_AREA)

def classer_par_sift_optimise(chemin_dossier):
    dossier = Path(chemin_dossier)
    if not dossier.is_dir():
        raise FileNotFoundError("Répertoire introuvable.")

    dossier_resultats = dossier / "Fonds_Tries"
    dossier_resultats.mkdir(exist_ok=True)

    fichiers = [f for f in dossier.glob("*.png") if "Fonds_Tries" not in f.parts]

    # Suppression des doublons exacts
    hash_vus = set()
    fichiers_uniques = []
    for f in fichiers:
        h = calculer_empreinte_md5(f)
        if h in hash_vus:
            f.unlink()
        else:
            hash_vus.add(h)
            fichiers_uniques.append(f)

    sift = cv2.SIFT_create()
    paysages = []
    portraits = []

    # Extraction optimisée
    for f in fichiers_uniques:
        img = cv2.imread(str(f), cv2.IMREAD_GRAYSCALE)
        if img is None:
            continue
        
        hauteur_orig, largeur_orig = img.shape
        
        # Redimensionnement avant SIFT
        img_reduite = redimensionner_image(img, max_dim=800)
        _, descripteurs = sift.detectAndCompute(img_reduite, None)
        
        donnees = {"chemin": f, "desc": descripteurs}
        
        # Le classement Paysage/Portrait conserve les dimensions d'origine
        if largeur_orig >= hauteur_orig:
            paysages.append(donnees)
        else:
            portraits.append(donnees)

    # Paramètres FLANN
    index_params = dict(algorithm=1, trees=5)
    search_params = dict(checks=50)
    flann = cv2.FlannBasedMatcher(index_params, search_params)

    compteur_paire = 1

    # Appariement
    for p in paysages:
        if p["desc"] is None or len(p["desc"]) < 2:
            continue

        meilleur_match = None
        max_correspondances = 0

        for po in portraits:
            if po["desc"] is None or len(po["desc"]) < 2:
                continue

            try:
                matches = flann.knnMatch(p["desc"], po["desc"], k=2)
            except cv2.error:
                continue

            # Application du Ratio de Lowe
            valides = sum(1 for match_tuple in matches if len(match_tuple) == 2 and match_tuple[0].distance < 0.7 * match_tuple[1].distance)

            if valides > max_correspondances:
                max_correspondances = valides
                meilleur_match = po

        # Validation de la paire
        if meilleur_match is not None and max_correspondances > 15:
            nouveau_nom_pa = dossier_resultats / f"{compteur_paire:03d}_pa.png"
            nouveau_nom_po = dossier_resultats / f"{compteur_paire:03d}_po.png"
            
            p["chemin"].rename(nouveau_nom_pa)
            meilleur_match["chemin"].rename(nouveau_nom_po)
            
            portraits.remove(meilleur_match)
            compteur_paire += 1

In [ ]:
# Exécution de la fonction
# Remplacer par le chemin cible
# classer_par_sift(r"C:\Chemin\Vers\Le\Dossier")


chemin_dossier = r"C:\Users\FLORIAN\Pictures\FDE"
classer_par_sift(chemin_dossier)

In [3]:
import os
import torch
import torch.nn.functional as F
from torchvision import models, transforms
from PIL import Image
from pathlib import Path

def initialiser_compteur(dossier_resultats):
    # Chercher le plus grand numéro existant pour ne pas écraser le travail
    fichiers_existants = list(dossier_resultats.glob("*_pa.png"))
    if not fichiers_existants:
        return 1
    indices = [int(f.stem.split('_')[0]) for f in fichiers_existants]
    return max(indices) + 1

def extraire_vecteurs(fichiers, modele, transform, device):
    # Générer les embeddings CNN pour une liste d'images
    vecteurs = []
    chemins_valides = []
    
    with torch.no_grad():
        for fichier in fichiers:
            try:
                img = Image.open(fichier).convert('RGB')
                tensor = transform(img).unsqueeze(0).to(device)
                vecteur = modele(tensor)
                vecteurs.append(vecteur.squeeze(0).cpu())
                chemins_valides.append(fichier)
            except Exception:
                continue
                
    if not vecteurs:
        return None, []
    return torch.stack(vecteurs), chemins_valides

def classer_fonds_cnn(chemin_dossier):
    dossier = Path(chemin_dossier)
    dossier_resultats = dossier / "Fonds_Tries"
    dossier_resultats.mkdir(exist_ok=True)

    # Reprendre à la suite du travail déjà effectué
    compteur_paire = initialiser_compteur(dossier_resultats)

    fichiers = [f for f in dossier.glob("*.png") if "Fonds_Tries" not in f.parts]

    paysages_chemins = []
    portraits_chemins = []

    # Séparation initiale basée sur les dimensions
    for f in fichiers:
        try:
            with Image.open(f) as img:
                largeur, hauteur = img.size
                if largeur >= hauteur:
                    paysages_chemins.append(f)
                else:
                    portraits_chemins.append(f)
        except Exception:
            pass

    # Configuration du modèle ResNet18 pré-entraîné
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    modele = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    modele.fc = torch.nn.Identity() # Retirer la couche de classification finale
    modele = modele.to(device)
    modele.eval()

    # Transformation standard pour ResNet
    transform = transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

    # Extraction des vecteurs de caractéristiques
    vecteurs_pa, chemins_pa = extraire_vecteurs(paysages_chemins, modele, transform, device)
    vecteurs_po, chemins_po = extraire_vecteurs(portraits_chemins, modele, transform, device)

    if vecteurs_pa is None or vecteurs_po is None:
        return

    # Normalisation L2 pour le calcul de la similarité cosinus
    vecteurs_pa = F.normalize(vecteurs_pa, p=2, dim=1)
    vecteurs_po = F.normalize(vecteurs_po, p=2, dim=1)

    # Calcul de la matrice de similarité complète (Produit matriciel)
    matrice_similarite = torch.mm(vecteurs_pa, vecteurs_po.t())

    # Association gloutonne (Greedy matching)
    portraits_disponibles = set(range(len(chemins_po)))

    for i, chemin_pa in enumerate(chemins_pa):
        if not portraits_disponibles:
            break
            
        similarites_ligne = matrice_similarite[i].clone()
        
        # Ignorer les portraits déjà associés
        for j in range(len(chemins_po)):
            if j not in portraits_disponibles:
                similarites_ligne[j] = -1.0
                
        meilleur_score, meilleur_index = torch.max(similarites_ligne, dim=0)
        meilleur_index = meilleur_index.item()

        # Seuil de similarité (0.8 est généralement robuste pour des images liées)
        if meilleur_score.item() > 0.8:
            nouveau_nom_pa = dossier_resultats / f"{compteur_paire:03d}_pa.png"
            nouveau_nom_po = dossier_resultats / f"{compteur_paire:03d}_po.png"
            
            chemin_pa.rename(nouveau_nom_pa)
            chemins_po[meilleur_index].rename(nouveau_nom_po)
            
            portraits_disponibles.remove(meilleur_index)
            compteur_paire += 1

# classer_fonds_cnn(r"C:\Chemin\Vers\Le\Dossier")

In [4]:
chemin_dossier = r"C:\Users\FLORIAN\Pictures\FDE"

classer_fonds_cnn(chemin_dossier)


Re index et delete les doublons

In [5]:
import hashlib
from pathlib import Path

def nettoyer_et_reindexer_fonds(chemin_dossier):
    dossier = Path(chemin_dossier)
    if not dossier.is_dir():
        return

    fichiers = list(dossier.glob("*.png"))
    paires = {}
    
    # 1. Groupement des fichiers par index
    for f in fichiers:
        parties = f.stem.split('_')
        if len(parties) != 2:
            continue
            
        index_str, type_img = parties
        if not index_str.isdigit() or type_img not in ['pa', 'po']:
            continue
            
        index = int(index_str)
        if index not in paires:
            paires[index] = {}
        paires[index][type_img] = f

    hash_vus = set()
    indices_valides = sorted(paires.keys())
    paires_a_conserver = []

    # 2. Suppression des orphelins et des doublons stricts
    for index in indices_valides:
        paire = paires[index]
        
        # Suppression des fichiers sans paire
        if 'pa' not in paire or 'po' not in paire:
            for f in paire.values():
                f.unlink()
            continue

        # Lecture par blocs pour calculer le hachage MD5 du paysage
        h = hashlib.md5()
        with open(paire['pa'], "rb") as f:
            for bloc in iter(lambda: f.read(4096), b""):
                h.update(bloc)
        empreinte = h.hexdigest()

        # Suppression des deux fichiers si la paire existe déjà
        if empreinte in hash_vus:
            paire['pa'].unlink()
            paire['po'].unlink()
        else:
            hash_vus.add(empreinte)
            paires_a_conserver.append(paire)

    # 3. Réindexation séquentielle sans collision
    for nouvel_index, paire in enumerate(paires_a_conserver, start=1):
        chemin_pa = paire['pa']
        chemin_po = paire['po']
        
        nouveau_nom_pa = dossier / f"{nouvel_index:03d}_pa.png"
        nouveau_nom_po = dossier / f"{nouvel_index:03d}_po.png"
        
        # Renommage uniquement si l'index doit être modifié
        if chemin_pa != nouveau_nom_pa:
            chemin_pa.rename(nouveau_nom_pa)
        if chemin_po != nouveau_nom_po:
            chemin_po.rename(nouveau_nom_po)

# Exécution 
# nettoyer_et_reindexer_fonds(r"C:\Chemin\Vers\Fonds_Tries")

In [7]:
chemin_dossier = r"C:\Users\FLORIAN\Pictures\FDE\Fonds_Tries"
nettoyer_et_reindexer_fonds(chemin_dossier)

In [8]:
import pathlib

def lister_dossiers_notebook(chemin_absolu):
    """Affiche les dossiers contenus dans le chemin spécifié."""
    # Créer un objet Path à partir du chemin
    p = pathlib.Path(chemin_absolu)
    
    # Vérifier l'existence et si c'est bien un répertoire
    if not p.exists() or not p.is_dir():
        print(f"Le chemin {chemin_absolu} est invalide ou n'est pas un dossier.")
        return

    # Extraire uniquement les dossiers (non récursif)
    dossiers = [f.name for f in p.iterdir() if f.is_dir()]
    
    # Affichage des résultats
    if dossiers:
        print(f"Dossiers trouvés dans {chemin_absolu} :")
        for nom in sorted(dossiers):
            print(f" - {nom}")
    else:
        print("Aucun dossier trouvé.")


In [9]:

# Exemple d'appel
lister_dossiers_notebook(r"M:\documents\Machine_Deep_Learning\Ebook\Packt Ebook HTML")

Dossiers trouvés dans M:\documents\Machine_Deep_Learning\Ebook\Packt Ebook HTML :
 - Autre Julia Docker Kubernetes
 - Autres
 - Base de Données SQL
 - Cloud BigData
 - Computer Vision
 - Data Engineering
 - Deep Learning
 - Ensemble Learning - Tuning
 - Geospatial
 - Graphs
 - Introduction à
 - Linux
 - ML DL Interprétation
 - ML Ops
 - Machine Engineering
 - Machine Learning
 - Mobile
 - Modèles Generatifs (GAN...)
 - NLP NLU
 - Network Data Mining
 - Preprocessing and Engeniering
 - Projets
 - Python
 - R
 - Recomandation
 - Reinforcement Learning
 - Rust
 - Scala
 - Spark Hadoop
 - Statistics Mathématiques
 - Time Series
 - Visualisation
 - WEB


In [10]:

# Exemple d'appel
lister_dossiers_notebook(r"C:\Users\FLORIAN\Downloads\Ebooks\Data_Science_book")

Dossiers trouvés dans C:\Users\FLORIAN\Downloads\Ebooks\Data_Science_book :
 - AI Engineering
 - Building AI Agents with LLMs, RAG, and Knowledge Graphs
 - Building Applications with AI Agents
 - Data Science at the Command Line, 2nd Edition
 - Deep Reinforcement Learning with Python With PyTorch, TensorFlow and OpenAI Gym
 - Essential Math for Data Science
 - Fuzzy Data Matching with SQL
 - Generative AI with LangChain - Second Edition
 - Hands-On Intelligent Agents with OpenAI Gym
 - Hands-On Machine Learning with Scikit-Learn and PyTorch
 - Hugging Face in Action
 - Implementing Data Mesh
 - Knowledge Graphs and LLMs in Action
 - LLMOps
 - LangChain for Life Sciences and Healthcare
 - Learning LangChain
 - Learning Snowflake SQL and Scripting
 - Practical Statistics for Data Scientists, 2nd Edition
 - Scaling Machine Learning with Spark
 - Scaling Python with Dask
 - Scaling Python with Ray
 - Streaming Databases
